## Example 1: Basic Vector Creation

In [ ]:
import pgvector
import numpy as np

# Example 1: Basic vector creation
print("Example 1 - Vector creation")

# Create from list
vec1 = pgvector.Vector([1.0, 2.0, 3.0])
print(f"From list: {vec1}")

# Create from numpy array
arr = np.array([4.0, 5.0, 6.0])
vec2 = pgvector.Vector(arr)
print(f"From numpy: {vec2}")

# Vector representation
print(f"Vector 1: {repr(vec1)}")
print(f"Vector 2: {str(vec2)}")
assert vec1.to_list() == [1.0, 2.0, 3.0], vec1.to_list()
assert vec2.to_list() == [4.0, 5.0, 6.0], vec2.to_list()
assert vec1.dimensions() == 3
assert repr(vec1) == "Vector([1.0, 2.0, 3.0])", repr(vec1)

# Text form used by PostgreSQL, and back
assert vec1.to_text() == "[1.0,2.0,3.0]", vec1.to_text()
assert pgvector.Vector.from_text(vec1.to_text()) == vec1

print("Example 1 completed")

## Example 2: Vector Conversion

In [ ]:
import pgvector
import numpy as np

# Example 2: Vector from numpy
print("Example 2 - Numpy integration")

# Create numpy array
data = np.array([1.5, -2.0, 3.14, 0.5])
print(f"Numpy array: {data}")

# Create vector from numpy
vec = pgvector.Vector(data)
print(f"pgvector: {vec}")

# Values are stored as 32-bit floats
assert vec.dimensions() == 4
assert vec.to_numpy().dtype == np.float32, vec.to_numpy().dtype
assert np.allclose(vec.to_numpy(), data), vec.to_numpy()

# Binary wire format round trip
restored = pgvector.Vector.from_binary(vec.to_binary())
assert restored == vec, (restored, vec)
print(f"Binary round trip: {restored}")

print("Example 2 completed")

## Example 3: Half-Precision, Sparse and Bit Vectors

In [ ]:
import pgvector

# Example 3: Half-precision, sparse and bit vectors
# (distances such as <-> and <=> are computed inside PostgreSQL; this package converts the values)
print("Example 3 - Other vector types")

half = pgvector.HalfVector([1.5, 2.0, 3.0])
print(f"HalfVector: {half.to_text()}")
assert half.to_list() == [1.5, 2.0, 3.0], half.to_list()
assert pgvector.HalfVector.from_text(half.to_text()) == half

# Sparse vectors keep only the non-zero entries; text form is {index:value}/dimensions with 1-based indices
sparse = pgvector.SparseVector({0: 1.0, 2: 3.0}, 4)
print(f"SparseVector: {sparse.to_text()}")
assert sparse.dimensions() == 4
assert sparse.indices() == [0, 2] and sparse.values() == [1.0, 3.0]
assert sparse.to_list() == [1.0, 0.0, 3.0, 0.0], sparse.to_list()
assert sparse.to_text() == "{1:1.0,3:3.0}/4", sparse.to_text()
assert pgvector.SparseVector.from_text("{1:1,3:2}/4").to_list() == [1.0, 0.0, 2.0, 0.0]

bits = pgvector.Bit([True, False, True])
print(f"Bit: {bits.to_text()}")
assert bits.to_text() == "101", bits.to_text()
assert bits.to_list() == [True, False, True]

print("Example 3 completed")


## Example 4: Vector Operations

In [ ]:
import pgvector
import numpy as np

# Example 4: Vector operations
print("Example 4 - Operations")

# Create numpy arrays for operations
arr1 = np.array([2.0, 4.0, -1.0])
arr2 = np.array([1.0, -2.0, 3.0])

# Addition
result_add = arr1 + arr2
vec_add = pgvector.Vector(result_add)
print(f"Addition: {vec_add}")

# Scalar multiplication
result_mult = arr1 * 2.5
vec_mult = pgvector.Vector(result_mult)
print(f"Scalar mult: {vec_mult}")

# Normalization
normalized = arr1 / np.linalg.norm(arr1)
vec_norm = pgvector.Vector(normalized)
print(f"Normalized: {vec_norm}")
assert vec_add.to_list() == [3.0, 2.0, 2.0], vec_add.to_list()
assert vec_mult.to_list() == [5.0, 10.0, -2.5], vec_mult.to_list()
assert abs(np.linalg.norm(vec_norm.to_numpy()) - 1.0) < 1e-6

print("Example 4 completed")

## Example 5: High-Dimensional Vectors

In [ ]:
import pgvector
import numpy as np

# Example 5: High-dimensional vectors
print("Example 5 - High dimensions")

# Test different dimensions
dims = [128, 256, 512, 1024]

for dim in dims:
    data = np.random.random(dim)
    vec = pgvector.Vector(data)
    print(f"Dim {dim}: created vector")
    assert vec.dimensions() == dim
    assert np.allclose(vec.to_numpy(), data, rtol=1e-6)
    assert pgvector.Vector.from_binary(vec.to_binary()) == vec

# Similarity search simulation
embedding_dim = 384
query = np.random.random(embedding_dim)
doc = np.random.random(embedding_dim)

query_vec = pgvector.Vector(query)
doc_vec = pgvector.Vector(doc)

# Cosine similarity of the stored (float32) values, computed with NumPy
q, d = query_vec.to_numpy(), doc_vec.to_numpy()
similarity = float(np.dot(q, d) / (np.linalg.norm(q) * np.linalg.norm(d)))
print(f"\nSimilarity: {similarity:.4f}")
assert query_vec.dimensions() == doc_vec.dimensions() == embedding_dim
assert 0.0 < similarity <= 1.0 + 1e-6, similarity  # both vectors have only non-negative entries

print("Example 5 completed")